# 03 - Exploratory Data Analysis (EDA)

Statistical analysis + visualizations answering the core business questions.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

from data_loader import load_processed_dataset
from analysis import (
    compute_basic_statistics,
    category_analysis,
    priority_analysis,
    status_analysis,
    product_analysis,
    channel_analysis,
    time_analysis,
    recurring_problems,
    business_insights,
    format_basic_stats_text,
)

In [ ]:
df = load_processed_dataset()
print(f'Cleaned dataset shape: {df.shape}')

In [ ]:
basic = compute_basic_statistics(df)
print(format_basic_stats_text(basic))

In [ ]:
cat_df = category_analysis(df)
cat_df

In [ ]:
pri_df = priority_analysis(df)
pri_df

In [ ]:
status_df = status_analysis(df)
status_df

In [ ]:
prod_df = product_analysis(df)
prod_df.head(10)

In [ ]:
chan_df = channel_analysis(df)
chan_df

In [ ]:
time_info = time_analysis(df)
if 'by_month' in time_info:
    display(pd.DataFrame(time_info['by_month']))
if 'date_range' in time_info:
    print('Date range:', time_info['date_range'])

In [ ]:
recurring = recurring_problems(df, top_n=10)
print('Top subjects:')
for s in recurring.get('top_subjects', []):
    print(f"  {s['count']:>5}  {s['subject']}")
print('\nTop keywords:')
for s in recurring.get('top_keywords', []):
    print(f"  {s['count']:>5}  {s['word']}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

if len(cat_df):
    sns.barplot(data=cat_df, x='ticket_count', y='Ticket Type', ax=axes[0], palette='viridis')
    axes[0].set_title('Ticket Count by Type')

if len(pri_df):
    sns.barplot(data=pri_df, x='Ticket Priority', y='ticket_count', ax=axes[1], palette='magma')
    axes[1].set_title('Priority Distribution')

if len(status_df):
    axes[2].pie(status_df['ticket_count'], labels=status_df['Ticket Status'], autopct='%1.1f%%', startangle=90)
    axes[2].set_title('Ticket Status Distribution')

rth = df.get('resolution_time_hours', pd.Series(dtype=float)).dropna()
if len(rth):
    sns.histplot(rth, bins=20, kde=True, ax=axes[3], color='teal')
    axes[3].set_title('Resolution Time (hours) Distribution')
    axes[3].set_xlabel('Hours')

plt.tight_layout()
out_png = os.path.abspath(os.path.join(os.getcwd(), '..', 'outputs', 'figures', 'eda_overview.png'))
os.makedirs(os.path.dirname(out_png), exist_ok=True)
plt.savefig(out_png, dpi=140, bbox_inches='tight')
print(f'Saved: {out_png}')
plt.show()

In [ ]:
insights = business_insights(df, basic, cat_df, pri_df, prod_df, time_info, recurring)
print(insights)